# Implementação do Multi-Layer Perceptron em R

In [1]:
# Iniciar o Colab com R
# https://colab.research.google.com/#create=true&language=r



Usando a função logistica como função de ativação

Razões:

1. Introduz não-linearidade
2. Produz saída entre 0 e 1 (probabilidade)
3. Tem derivada simples, facilitando o backpropagation

In [2]:
#a função de ativação 
funcao.ativacao <- function(v){
    #funcao logistica
    y <- 1/(1+exp(-v))

    return(y)
}

In [3]:
#implementando a derivada da função logistica
der.funcao.ativacao <- function(y){
    derivada <- y * (1-y)

    return(derivada)
}

In [4]:
#criando uma arquitetura para a MLP
arquitetura <- function(num.entrada, num.escondida, num.saida,
                        funcao.ativacao, der.funcao.ativacao) {
    arq <- list()

    #parametro da rede
    arq$num.entrada <- num.entrada
    arq$num.escondida <- num.escondida
    arq$num.saida <- num.saida
    arq$funcao.ativacao <- funcao.ativacao
    arq$der.funcao.ativacao <- der.funcao.ativacao

    # 2 neuronios na camada escondida
    #
    #       Ent1    Ent2   Bias
    # 1     w11     w12    w13
    # 2     w21     w22    w23

    # Pesos conectando entrada e escondida
    #adicionamos +1 por conta dos pesos do bias
    num.pesos.entrada.escondida <- (num.entrada+1)*num.escondida
    arq$escondida <- matrix(runif(min=-0.5, max=0.5, num.pesos.entrada.escondida),
                                nrow = num.escondida, ncol=num.entrada+1)

    #Pesos conectando escondida e saida
    num.pesos.escondida.saida <- (num.escondida+1)*num.saida
    arq$saida <- matrix(runif(min=-0.5,max=0.5, num.pesos.escondida.saida),
                            nrow = num.saida, ncol = num.escondida+1)

    return(arq)

}

fase de propagação

In [5]:
mlp.propagacao <- function(arq, dados.exemplos){

    #entrada -> camada escondida
    #multiplicacao de matriz : %*%
    v.entrada.escondida <- arq$escondida %*% as.numeric(c(dados.exemplos, 1))
    y.entrada.escondida <- arq$funcao.ativacao(v.entrada.escondida)

    #camada escondida -> camada de saida
    v.escondida.saida <- arq$saida %*% c(y.entrada.escondida, 1)
    y.escondida.saida <- arq$funcao.ativacao(v.escondida.saida)

    #resultados
    resultados <- list()
    resultados$v.entrada.escondida <- v.entrada.escondida
    resultados$y.entrada.escondida <- y.entrada.escondida
    resultados$v.escondida.saida <- v.escondida.saida
    resultados$y.escondida.saida <- y.escondida.saida

    return(resultados)
}

**Backpropagation**

 O que é: Um método supervisionado de aprendizado que utiliza o gradiente descendente para otimizar pesos e vieses (bias).

Objetivo: Minimizar a função de custo (erro) entre a saída prevista e a saída real.

Importância: Sem o backpropagation, redes neurais multicamadas não conseguiriam ajustar eficientemente as camadas ocultas

Funcionamento:

**Forward Pass (Propagação Direta)**: Dados de entrada passam pela rede, camadas ocultas processam e geram uma saída.

**Cálculo do Erro:** A saída da rede é comparada com a "verdade básica" (saída desejada), gerando uma função de perda.

**Backward Pass (Retropropagação):** O erro é propagado de volta. O algoritmo calcula a derivada parcial da função de custo em relação a cada peso, utilizando a regra da cadeia do cálculo.

**Atualização dos Pesos:** Os pesos são ajustados (subtraindo o gradiente multiplicado por uma taxa de aprendizado) para reduzir o erro na próxima iteração.

In [6]:
mlp.retropropagacao <- function(arq, dados, n, limiar){

    erroQuadratico <- 2 * limiar
    epocas <- 0

    #O treinamento ocorre enquanto o erroQuad > limiar
    while(erroQuadratico > limiar){
        erroQuadratico <- 0

        #Treino para todos os exemplos(i.e epoca)
        for(i in 1:nrow(dados)){
            #pega um exemplo de entrada
            x.entrada <- dados[i,1:arq$num.entrada]
            x.saida <- dados[i, ncol(dados)]

            #Foward Pass
            #pega a saida da rede para o exemplo
            result <- mlp.propagacao(arq, x.entrada)
            y <- result$y.escondida.saida

            #Calculo do erro
            erro <- x.saida - y

            #soma erro quadratico
            erroQuadratico <- erroQuadratico + erro*erro

            #Backward Pass

            # Gradiente local no neuronio de saida
            # erro * derivada da funcao de ativacao
            gradiente.local.saida <- erro * arq$der.funcao.ativacao(y)

            # Gradiente local no neuronio escondido
            # derivada da funcao de ativacao no neuronio escondido * soma dos gradientes
            # locais dos neuronios conectados na proxima camada * pesos conectando a camada
            # escondida com a saida
            pesos.saida <- arq$saida[,1:arq$num.escondida]
            gradiente.local.escondida <- as.numeric(arq$der.funcao.ativacao(result$y.entrada.escondida)) * (gradiente.local.saida %*% pesos.saida)

            #Atualizando os pesos sinapticos
            arq$saida <- arq$saida + n * (gradiente.local.saida %*% c(result$y.entrada.escondida, 1))

            #escondida
            arq$escondida <- arq$escondida + n * (t(gradiente.local.escondida) %*% as.numeric(c(x.entrada,1)))
        }

        erroQuadratico <- erroQuadratico / nrow(dados)
        cat("Erro Quadratico Medio = ", erroQuadratico, "\n")
        flush.console()
        epocas <- epocas+1
    }

    retorno <- list()
    retorno$arq <- arq
    retorno$epocas <- epocas

    return(retorno)
}

In [7]:
# Leitura de uma tabela com os dados do problema XOR
dados <- read.table('XOR.txt')
print(dados)

  V1 V2 V3
1  0  0  0
2  0  1  1
3  1  0  1
4  1  1  0


In [8]:
# Nossa arquitetura
arq <- arquitetura(2,2,1,funcao.ativacao, der.funcao.ativacao)
print(arq)

$num.entrada
[1] 2

$num.escondida
[1] 2

$num.saida
[1] 1

$funcao.ativacao
<srcref: file "" chars 2:20 to 7:1>

$der.funcao.ativacao
<srcref: file "" chars 2:24 to 6:1>

$escondida
          [,1]        [,2]       [,3]
[1,] 0.3778651 -0.24078653 -0.3652943
[2,] 0.3223562  0.02926528  0.2216363

$saida
          [,1]       [,2]      [,3]
[1,] 0.4726642 -0.4894914 0.2526804



In [9]:
# Vamos treinar nossa MLP
#taxa de aprendizado: 0.1
#limiar: 0.001
modelo <- mlp.retropropagacao(arq,dados,0.1,1e-3)
print(modelo)

Erro Quadratico Medio =  0.2539981 
Erro Quadratico Medio =  0.2538812 
Erro Quadratico Medio =  0.253773 
Erro Quadratico Medio =  0.2536729 
Erro Quadratico Medio =  0.2535801 
Erro Quadratico Medio =  0.2534944 
Erro Quadratico Medio =  0.253415 


Erro Quadratico Medio =  0.2533415 
Erro Quadratico Medio =  0.2532735 
Erro Quadratico Medio =  0.2532106 
Erro Quadratico Medio =  0.2531525 
Erro Quadratico Medio =  0.2530987 
Erro Quadratico Medio =  0.2530489 
Erro Quadratico Medio =  0.2530028 
Erro Quadratico Medio =  0.2529603 
Erro Quadratico Medio =  0.2529209 
Erro Quadratico Medio =  0.2528845 
Erro Quadratico Medio =  0.2528508 
Erro Quadratico Medio =  0.2528196 
Erro Quadratico Medio =  0.2527908 
Erro Quadratico Medio =  0.2527642 
Erro Quadratico Medio =  0.2527395 
Erro Quadratico Medio =  0.2527167 
Erro Quadratico Medio =  0.2526956 
Erro Quadratico Medio =  0.2526761 
Erro Quadratico Medio =  0.2526581 
Erro Quadratico Medio =  0.2526414 
Erro Quadratico Medio =  0.252626 
Erro Quadratico Medio =  0.2526117 
Erro Quadratico Medio =  0.2525985 
Erro Quadratico Medio =  0.2525863 
Erro Quadratico Medio =  0.252575 
Erro Quadratico Medio =  0.2525645 
Erro Quadratico Medio =  0.2525548 
Erro Quadratico Medio =  0.252

In [10]:
#Testando em dados novos
retorno <- mlp.propagacao(modelo$arq,c(0,1))
print(retorno$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(1,0))$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(1,1))$y.escondida.saida)

print(mlp.propagacao(modelo$arq,c(0,0))$y.escondida.saida)

          [,1]
[1,] 0.9637583
          [,1]
[1,] 0.9703739
           [,1]
[1,] 0.02814399
          [,1]
[1,] 0.0318687


In [11]:
# Leitura de dados sobre risco de crédito.
# Queremos prever, com base nas variáveis de entrada, se ocorrerá ou não um calote em 10 anos

dataset <- read.csv("creditset.csv")
dim(dataset)
head(dataset)

# clientid: identificação do cliente
# income: renda anual
# age: idade
# loan: empréstimo com data de no mínimo 10 anos atrás
# LTI: razão entre valor do empréstimo e renda anual
# default10yr: se ocorrerá (1) ou não (0) um calote em 10 anos

[1] 2000    6

,clientid,income,age,loan,LTI,default10yr
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,1,66155.93,59.01702,8106.5321,0.1225367512,0
2,2,34415.15,48.11715,6564.7450,0.1907515807,0
3,3,57317.17,63.10805,8020.9533,0.1399397997,0
4,4,42709.53,45.75197,6103.6423,0.1429105321,0
5,5,66952.69,18.58434,8770.0992,0.1309895000,1
6,6,24904.06,57.47161,15.4986,0.0006223321,0


In [12]:
# Vamos utilizar como variáveis de entrada LTI e age
dados <- dataset[,c(3,5,6)]
dados[,1:2] <- scale(dados[,1:2]) # normalização dos dados (media 0 e desvio 1)
head(dados)

,age,LTI,default10yr
,<dbl>,<dbl>,<int>
1,1.3639917,0.4188448,0
2,0.5421329,1.6027146,0
3,1.6724592,0.7208751,0
4,0.3637962,0.7724322,0
5,-1.6846667,0.5655424,1
6,1.2474667,-1.6969826,0


In [13]:
# Vamos escolher aleatoriamente dados para treino e teste.
# O conjunto de dados já está randomizado.
# Assim, Vamos pegar os primeiros 1400 exemplos para treino e o restante para teste
dados.treino <- dados[1:1400,]
dados.teste <- dados[1401:2000,]

In [14]:
# Vamos utilizar como variáveis de entrada LTI e age
dados <- dataset[,c(3,5,6)]
dados[,1:2] <- scale(dados[,1:2]) # normalização dos dados (media 0 e desvio 1)
head(dados)

,age,LTI,default10yr
,<dbl>,<dbl>,<int>
1,1.3639917,0.4188448,0
2,0.5421329,1.6027146,0
3,1.6724592,0.7208751,0
4,0.3637962,0.7724322,0
5,-1.6846667,0.5655424,1
6,1.2474667,-1.6969826,0


In [15]:
# Vamos treinar nossa rede com 4 neurônios na camada escondida
modelo <- mlp.retropropagacao(arq,dados.treino,0.3,1e-3)
print(modelo)

Erro Quadratico Medio =  0.09022748 
Erro Quadratico Medio =  0.04884193 
Erro Quadratico Medio =  0.03851232 
Erro Quadratico Medio =  0.03514291 
Erro Quadratico Medio =  0.03365 
Erro Quadratico Medio =  0.03285449 
Erro Quadratico Medio =  0.03236515 
Erro Quadratico Medio =  0.03200798 
Erro Quadratico Medio =  0.03167479 
Erro Quadratico Medio =  0.03126821 
Erro Quadratico Medio =  0.03068287 
Erro Quadratico Medio =  0.0298133 
Erro Quadratico Medio =  0.02857201 
Erro Quadratico Medio =  0.02690254 
Erro Quadratico Medio =  0.02481879 
Erro Quadratico Medio =  0.02247239 
Erro Quadratico Medio =  0.02013302 
Erro Quadratico Medio =  0.0180362 
Erro Quadratico Medio =  0.01626979 
Erro Quadratico Medio =  0.01481241 
Erro Quadratico Medio =  0.01361361 
Erro Quadratico Medio =  0.01262419 
Erro Quadratico Medio =  0.01180064 
Erro Quadratico Medio =  0.01110543 
Erro Quadratico Medio =  0.01050775 
Erro Quadratico Medio =  0.009984149 
Erro Quadratico Medio =  0.009517924 
Erro

In [16]:
# Fazendo predicoes para cada exemplo de teste
predicoes <- vector()
for(i in 1:nrow(dados.teste)){

  pred <- mlp.propagacao(modelo$arq,dados.teste[i,1:2])$y.escondida.saida

  predicoes <- c(predicoes,pred)
}
print(predicoes)

  [1] 5.704904e-05 6.113333e-13 6.113329e-13 5.704904e-05 5.704904e-05
  [6] 6.113329e-13 6.113329e-13 7.406886e-13 5.704904e-05 9.944868e-09
 [11] 4.799102e-05 4.799102e-05 6.113329e-13 6.113329e-13 9.875349e-01
 [16] 4.799065e-05 6.113331e-13 3.128489e-06 6.113347e-13 6.113329e-13
 [21] 6.113417e-13 5.704798e-05 3.250007e-10 6.113389e-13 5.704884e-05
 [26] 6.185775e-13 6.137769e-13 6.113329e-13 6.113528e-13 9.997223e-01
 [31] 6.113329e-13 6.113329e-13 6.113329e-13 5.188676e-11 1.276570e-11
 [36] 9.997766e-01 6.113329e-13 4.799102e-05 6.113329e-13 4.799102e-05
 [41] 9.997768e-01 4.098795e-11 6.113329e-13 9.997768e-01 6.113331e-13
 [46] 6.113329e-13 5.704904e-05 3.870233e-12 6.113329e-13 6.477126e-13
 [51] 4.799169e-05 6.277353e-13 6.182763e-07 9.997768e-01 5.704606e-05
 [56] 6.471193e-13 6.113329e-13 6.113410e-13 5.704459e-05 4.797696e-05
 [61] 5.704936e-05 5.704904e-05 6.987097e-04 1.409935e-05 9.997312e-01
 [66] 4.799102e-05 7.799500e-05 5.704796e-05 1.975004e-05 5.702518e-05
 [71] 

In [17]:
# Criando uma matriz para comparação dos resultados
matriz.comparacao <- cbind(dados.teste[,3],predicoes)
colnames(matriz.comparacao) <- c('V','P')
print(matriz.comparacao)

       V            P
  [1,] 0 5.704904e-05
  [2,] 0 6.113333e-13
  [3,] 0 6.113329e-13
  [4,] 0 5.704904e-05
  [5,] 0 5.704904e-05
  [6,] 0 6.113329e-13
  [7,] 0 6.113329e-13
  [8,] 0 7.406886e-13
  [9,] 0 5.704904e-05
 [10,] 0 9.944868e-09
 [11,] 0 4.799102e-05
 [12,] 0 4.799102e-05
 [13,] 0 6.113329e-13
 [14,] 0 6.113329e-13
 [15,] 1 9.875349e-01
 [16,] 0 4.799065e-05
 [17,] 0 6.113331e-13
 [18,] 0 3.128489e-06
 [19,] 0 6.113347e-13
 [20,] 0 6.113329e-13
 [21,] 0 6.113417e-13
 [22,] 0 5.704798e-05
 [23,] 0 3.250007e-10
 [24,] 0 6.113389e-13
 [25,] 0 5.704884e-05
 [26,] 0 6.185775e-13
 [27,] 0 6.137769e-13
 [28,] 0 6.113329e-13
 [29,] 0 6.113528e-13
 [30,] 1 9.997223e-01
 [31,] 0 6.113329e-13
 [32,] 0 6.113329e-13
 [33,] 0 6.113329e-13
 [34,] 0 5.188676e-11
 [35,] 0 1.276570e-11
 [36,] 1 9.997766e-01
 [37,] 0 6.113329e-13
 [38,] 0 4.799102e-05
 [39,] 0 6.113329e-13
 [40,] 0 4.799102e-05
 [41,] 1 9.997768e-01
 [42,] 0 4.098795e-11
 [43,] 0 6.113329e-13
 [44,] 1 9.997768e-01
 [45,] 0 6

In [18]:
# Matriz de confusão com o arredondamento das predições
mc <- table(matriz.comparacao[,1],round(matriz.comparacao[,2]))
print(mc)

acc <- sum(diag(mc))/sum(mc)
print(acc)
sum(mc)
sum(diag(mc))

   
      0   1
  0 522   0
  1   0  78
[1] 1


[1] 600

[1] 600